# Neural audio codecs with ESPnet-Codec

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/codec_demo.ipynb)

Squeeze a waveform into a handful of integers per frame, then rebuild it.
Listen to what survives, and count what it cost.

CPU, and the models are small.


## Install


In [ ]:
%pip install -q "espnet==202610.post1" espnet_model_zoo librosa


## A few seconds of speech


In [ ]:
import librosa, torch
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("sample.wav", sr=16000)
speech = speech[: rate * 3]
print("original")
display(Audio(speech, rate=rate))


## Encode and decode

[`libritts_soundstream16k`](https://huggingface.co/espnet/libritts_soundstream16k)
returns both halves at once: `codes`, the integers it chose, and
`resyn_audio`, what those integers rebuild.


In [ ]:
from espnet2.bin.gan_codec_inference import AudioCoding

codec = AudioCoding.from_pretrained("espnet/libritts_soundstream16k", device="cpu")

out = codec(torch.tensor(speech).unsqueeze(0))
codes, resynthesised = out["codes"], out["resyn_audio"]
print(f"codes {tuple(codes.shape)}  (codebooks, batch, frames)")
print("resynthesised")
display(Audio(resynthesised.view(-1).cpu().numpy(), rate=rate))


## What it cost

One integer per codebook per frame. The bitrate follows from how many
codebooks there are, how many frames a second, and how many values an
integer can take.


In [ ]:
codebooks, _, frames = codes.shape
seconds = len(speech) / rate
values = int(codes.max()) + 1
bits = codebooks * (frames / seconds) * (values - 1).bit_length()

print(f"{codebooks} codebooks x {frames / seconds:.0f} frames/s x {(values - 1).bit_length()} bits")
print(f"{bits / 1000:.1f} kbit/s, against {rate * 16 / 1000:.0f} kbit/s for the 16-bit original")


## Where next

- **Other codecs**: `espnet/libritts_encodec_16k`, `espnet/libritts_dac_16k`
  and the `dac_16k_*_survey` models load the same way — swap the tag
- **The comparison**, with what happens when codebooks are dropped and with
  VERSA scores for each: [`../Courses/CMUSpeechTechnology26S/neural_codec.ipynb`](../Courses/CMUSpeechTechnology26S/neural_codec.ipynb)
